# OpenSource Clipping: vLLM + Gemma 4 Q4 di Kaggle

Preset **eksperimental**. Log Kaggle T4 sebelumnya menunjukkan satu eval transkrip pendek berhasil setelah workaround shared memory Triton. Pipeline lengkap, model alternatif, dan perubahan dependency berikut tetap perlu divalidasi di Kaggle. Jalankan notebook ini di **Linux x86_64 (kernel Python 3.13 juga boleh)**, Internet aktif, satu GPU kelas 16 GB dengan compute capability **SM 7.5+**. T4 memenuhi syarat arsitektur dan memerlukan workaround attention; **P100 tidak didukung**, walaupun VRAM-nya 16 GB. RAM sistem berbeda dari VRAM. Kecocokan arsitektur belum menjamin model dan KV cache muat.

Model: `google/gemma-4-E2B-it-qat-w4a16-ct`, checkpoint resmi QAT INT4 (W4A16), dengan revision dipin. Server vLLM dan aplikasi menggunakan **dua virtual environment terpisah**. Sel Prepare Python menyiapkan CPython 3.12 untuk setup kedua environment; kernel notebook tidak diganti. Lihat `docs/KAGGLE_VLLM.md` untuk batasan dan troubleshooting.

Jalankan sel berurutan. Setup mengunduh dependency; Start memuat model; Eval benar-benar melakukan inferensi. Notebook ini tidak ditujukan untuk dijalankan di PC lokal.

## 1. Siapkan checkout

Pilih branch/commit yang sudah memuat implementasi ini melalui `REPO_REF`. Checkout yang sudah ada dipertahankan dan tidak otomatis di-pull/reset. Perbarui checkout secara eksplisit bila masih memakai versi lama.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/NaufalRizqullah/opensource-clipping.git"
REPO_REF = "main"  # Ganti jika implementasi masih berada di branch lain.
workspace = Path("/kaggle/working")
if not workspace.is_dir():
    raise RuntimeError("Jalankan notebook ini di Kaggle, bukan di PC lokal.")
PROJECT_DIR = workspace / "opensource-clipping"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)
elif not (PROJECT_DIR / ".git").is_dir():
    raise RuntimeError("PROJECT_DIR sudah ada dan bukan checkout; pilih direktori lain.")
required = ["scripts/kaggle_storage.py", "scripts/kaggle_vllm.py", "scripts/kaggle_vllm_compat.py", "scripts/kaggle_python.py", "scripts/eval_local_llm.py", "requirements-vllm-kaggle.txt"]
if any(not (PROJECT_DIR / path).is_file() for path in required):
    raise RuntimeError("Perbarui checkout ke branch/commit implementasi vLLM sebelum melanjutkan.")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Checkout siap; file dan output yang sudah ada dipertahankan.")


## 2. Periksa GPU sebelum instalasi

Jika Kaggle menyediakan 2× T4, preset tetap menggunakan **satu GPU**, indeks 0. Dua GPU 16 GB tidak otomatis menjadi satu GPU 32 GB. CPU saja dan P100 dihentikan oleh preflight ini.

Preflight hanya membaca metadata GPU; Python kernel tidak harus 3.11/3.12. Environment aplikasi/server disiapkan setelah langkah ini.

In [ ]:
GPU_INDEX = 0
subprocess.run(
    [sys.executable, "scripts/kaggle_vllm.py", "preflight", "--gpu-index", str(GPU_INDEX)],
    cwd=PROJECT_DIR, check=True,
)


## 2b. Siapkan penyimpanan dan Python 3.12

Mode temporary menempatkan Python, environment Studio/vLLM dan unduhan sementara di luar output tersimpan. Jika ruang yang dilaporkan sedikit, pilih RUNTIME_ROOT pada filesystem lain yang dapat ditulis. Mode checkout memakai kembali lokasi environment lama. Simpan hasil dan log sebelum membuka sesi baru.

Profil Torch cpu menghemat ruang untuk Studio dengan Whisper CPU/int8 atau subtitle. Profil cuda menyediakan Torch GPU untuk Studio. Server vLLM memakai GPU pada kedua profil. Sel ini menggunakan CPython 3.12 yang tersedia atau mengunduhnya untuk setup.

In [ ]:
from scripts.kaggle_python import bootstrap_environment, ensure_python

from scripts.kaggle_storage import check_storage, prepare_storage

STORAGE_MODE = "temporary"  # "checkout" uses the original .cache paths
RUNTIME_ROOT = None  # Optional absolute path on a writable scratch disk
MIN_OUTPUT_GIB = 1.0  # Increase to the space needed for your source and clips
STUDIO_TORCH_PROFILE = "cuda"  # "cpu" saves disk; choose Whisper CPU/int8 or subtitles
if STUDIO_TORCH_PROFILE not in {"cuda", "cpu"}:
    raise ValueError("Choose cuda or cpu for STUDIO_TORCH_PROFILE.")

storage = prepare_storage(PROJECT_DIR, workspace, mode=STORAGE_MODE, runtime_root=RUNTIME_ROOT)
check_storage(storage, MIN_OUTPUT_GIB)
cache_env = storage.cache_environment()
PYTHON_CACHE = storage.python_cache
BOOTSTRAP_PYTHON = ensure_python(PYTHON_CACHE)
print(f"Kernel notebook: Python {sys.version_info.major}.{sys.version_info.minor}")
print(f"Python untuk setup: {BOOTSTRAP_PYTHON}")
subprocess.run(
    [str(BOOTSTRAP_PYTHON), "--version"],
    cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
)


## 3. Secrets dan konfigurasi

Terima ketentuan akses model pada Hugging Face jika diminta, lalu tambahkan `HF_TOKEN` di Kaggle **Add-ons → Secrets** dan aktifkan akses notebook. `NGROK_AUTHTOKEN` hanya diperlukan untuk Studio; `PEXELS_API_KEY` untuk B-roll opsional. Google/NVIDIA API key tidak diperlukan untuk vLLM.

API vLLM hanya dibuka di `127.0.0.1:8001`. Key internal dibuat di memori jika belum ada; nilainya tidak dicetak atau disimpan dalam notebook. Setelah mengganti key, hentikan server lama sebelum memulai ulang.

In [ ]:
from secrets import token_urlsafe

try:
    from kaggle_secrets import UserSecretsClient
    secret_reader = UserSecretsClient().get_secret
except Exception as exc:
    secret_reader = None
    print(f"Kaggle Secrets belum tersedia ({type(exc).__name__}).")

for name in ("HF_TOKEN", "NGROK_AUTHTOKEN", "PEXELS_API_KEY", "VLLM_API_KEY"):
    value = os.environ.get(name, "").strip()
    if not value and secret_reader is not None:
        try:
            value = (secret_reader(name) or "").strip()
        except Exception:
            pass
    if value:
        os.environ[name] = value
    print(f"{name}: {'configured' if value else 'not configured'}")
del value
if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN opsional untuk checkpoint publik; tambahkan jika download meminta autentikasi.")
if not os.environ.get("VLLM_API_KEY"):
    os.environ["VLLM_API_KEY"] = token_urlsafe(32)

VLLM_MODEL = "google/gemma-4-E2B-it-qat-w4a16-ct"
VLLM_PORT = 8001
CONTEXT_LIMIT = 16384
OUTPUT_TOKENS = 4096
os.environ.update({
    "VLLM_BASE_URL": f"http://127.0.0.1:{VLLM_PORT}",
    "VLLM_MODEL": VLLM_MODEL,
    "VLLM_MAX_TOKENS": str(OUTPUT_TOKENS),
    "VLLM_CONTEXT_LIMIT": str(CONTEXT_LIMIT),
    "VLLM_TIMEOUT": "900",
})
print("Konfigurasi vLLM tersimpan di environment proses; tidak ada key yang dicetak.")


## 4. Install aplikasi dan server secara terpisah

Sel ini dapat memakan waktu dan ruang disk cukup besar. Lingkungan aplikasi mempertahankan constraints Torch/NumPy/MediaPipe; vLLM 0.32.0 memakai wheel CUDA 12.9 dan Transformers 5.16.1 di lingkungan sendiri. CUDA wheel tetap harus cocok dengan driver sesi Kaggle. TorchCodec dipin ke wheel CPU untuk decoder media; inference vLLM tetap menggunakan GPU CUDA 12.9. Siapkan FFmpeg terlebih dahulu. Kesalahan resolver, `pip check`, atau pemeriksaan import native menghentikan setup. Log pemeriksaan: `logs/kaggle-vllm-import.log`. Pemeriksaan ini tidak memuat model atau memulai server.

Kedua setup dipanggil melalui BOOTSTRAP_PYTHON dari sel Prepare Python, bukan sys.executable milik kernel.

In [ ]:
import shutil

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
print("FFmpeg siap.")


In [ ]:
STUDIO_ENV = (storage.studio_env if STUDIO_TORCH_PROFILE == "cuda"
              else storage.root / "kaggle-studio-cpu" / "venv")
STUDIO_PYTHON = STUDIO_ENV / "bin/python"
VLLM_ENV = storage.vllm_env

subprocess.run(
    [str(BOOTSTRAP_PYTHON), "scripts/kaggle_studio.py", "setup", "--venv", str(STUDIO_ENV),
     "--torch-profile", STUDIO_TORCH_PROFILE],
    cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
)
check_storage(storage, MIN_OUTPUT_GIB)
subprocess.run(
    [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "setup", "--venv", str(VLLM_ENV)],
    cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
)


## 5. Start vLLM

Preset: FP16 activations, satu GPU, eager execution, maksimal satu sequence, text-only, thinking dimatikan. Quantization dibaca dari checkpoint QAT; tidak menggunakan quantizer tambahan. Unduhan pertama bisa lama. Helper memeriksa `/v1/models` dan hanya mengelola proses yang identitasnya terverifikasi.

Pada T4 (SM 7.5), Start memeriksa dan menerapkan workaround tile attention 16 di environment server vLLM 0.32.0 sebelum memuat model. Ini mengatasi jalur kernel yang meminta shared memory 96 KiB pada batas T4 64 KiB; verifikasi inference tetap dilakukan pada Eval. Versi/sumber yang berbeda ditolak, sumber asli dicadangkan, dan rerun tidak menumpuk patch.

Khusus error shared memory Triton sebelumnya, perbarui checkout ke commit perbaikan, lalu jalankan Stop → Start → Eval. Tidak perlu instal ulang dependency/model untuk patch attention saja. Jika error menyebut `libnvrtc.so.13` atau TorchCodec, jalankan ulang Setup sebelum Start agar wheel decoder CPU terpasang; simpan `logs/kaggle-vllm-import.log` jika pemeriksaan import gagal. Pastikan output Start memuat `T4 attention workaround applied` atau `already applied`. `/v1/models` merespons belum membuktikan inference berhasil.

In [ ]:
# Parser otomatis untuk Gemma 4 resmi dan Qwen3 resmi; lihat panduan untuk override.
subprocess.run(
    [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "start",
     "--venv", str(VLLM_ENV), "--gpu-index", str(GPU_INDEX),
     "--model", VLLM_MODEL, "--port", str(VLLM_PORT),
     "--context", str(CONTEXT_LIMIT), "--gpu-memory-utilization", "0.85"],
    cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
)


## 6. Eval transkrip pendek

Sel ini mengirim fixture 2 menit ke server vLLM. Tidak memuat Whisper atau merender video. Provider memeriksa token input + jatah output, schema, timestamp, durasi, dan overlap; kegagalan tidak dialihkan ke cloud. Hasil valid bukan bukti kualitas klip sudah setara Gemini.

In [ ]:
from scripts.kaggle_studio import runtime_environment

EVAL_OUTPUT = workspace / "vllm-eval-short.json"
evaluation = subprocess.run(
    [str(STUDIO_PYTHON), "scripts/eval_local_llm.py",
     "--transcript", "tests/fixtures/eval_transcript_short.txt",
     "--clips", "1", "--output", str(EVAL_OUTPUT)],
    cwd=PROJECT_DIR, env=runtime_environment(STUDIO_ENV) | cache_env, check=False,
)
print(f"Hasil eval: {EVAL_OUTPUT}")
if evaluation.returncode:
    server_log = PROJECT_DIR / "logs/kaggle-vllm.log"
    print(f"Log server: {server_log}")
    if server_log.is_file():
        print("\n".join(server_log.read_text(errors="replace").splitlines()[-120:]))
    raise RuntimeError("Eval gagal. Simpan JSON dan log server di atas sebelum Stop/Start.")


## 7. Regression tests di Kaggle (opsional)

Ubah `RUN_REGRESSION_TESTS` menjadi `True` untuk menjalankan suite offline. GPU, jaringan provider, dan instalasi oleh test dimock. Smoke test nyata memakai `RUN_LIVE_TEST=True` dan harus dijalankan setelah server siap. Tidak ada test yang dijalankan di PC lokal selama penyusunan implementasi.

In [ ]:
RUN_REGRESSION_TESTS = False
RUN_LIVE_TEST = False

if RUN_REGRESSION_TESTS or RUN_LIVE_TEST:
    subprocess.run(
        [str(STUDIO_PYTHON), "-m", "pip", "install",
         "-c", "constraints-kaggle.txt", "-r", "requirements-test.txt"],
        cwd=PROJECT_DIR, env=runtime_environment(STUDIO_ENV) | cache_env, check=True,
    )
if RUN_REGRESSION_TESTS:
    test_environment = runtime_environment(STUDIO_ENV) | cache_env
    test_environment.pop("RUN_LOCAL_LLM_TESTS", None)
    subprocess.run(
        [str(STUDIO_PYTHON), "-m", "pytest", "tests", "-q", "-m", "not local_llm"],
        cwd=PROJECT_DIR, env=test_environment, check=True,
    )
if RUN_LIVE_TEST:
    test_environment = runtime_environment(STUDIO_ENV) | cache_env
    test_environment["RUN_LOCAL_LLM_TESTS"] = "1"
    subprocess.run(
        [str(STUDIO_PYTHON), "-m", "pytest", "tests/test_vllm_live.py", "-q"],
        cwd=PROJECT_DIR, env=test_environment, check=True,
    )


## 8. Studio (opsional)

Ubah `START_STUDIO=True`, pastikan `NGROK_AUTHTOKEN` terpasang, lalu jalankan. Salin Public URL sesudah **BACKEND SERVER READY** ke Studio → Connect. Pada New Job, pilih **vLLM**, model preset di atas, satu klip dulu, **Whisper Device: CPU** dan model `small` untuk awal. UI mengirim compute type `int8` untuk kombinasi vLLM + CPU. vLLM masih menahan VRAM selama server hidup; menjalankan Whisper CUDA bersamaan dapat menyebabkan OOM.

Sel Studio menunggu sampai dihentikan. Ngrok hanya mengekspos backend Studio di port 8000, bukan server vLLM. Jangan membagikan URL tunnel. TTS edge-tts dan B-roll Pexels tetap membutuhkan jaringan.

In [ ]:
START_STUDIO = False

if START_STUDIO:
    if not os.environ.get("NGROK_AUTHTOKEN", "").strip():
        raise RuntimeError("Pasang NGROK_AUTHTOKEN, lalu ulangi sel konfigurasi.")
    from scripts.kaggle_studio import stop_process
    check_storage(storage, MIN_OUTPUT_GIB)
    supervisor = subprocess.Popen(
        [str(STUDIO_PYTHON), "scripts/kaggle_studio.py", "start",
         "--venv", str(STUDIO_ENV), "--port", "8000"],
        cwd=PROJECT_DIR, env=runtime_environment(STUDIO_ENV) | cache_env, start_new_session=True,
    )
    try:
        return_code = supervisor.wait()
        if return_code not in (0, 130):
            raise RuntimeError(f"Studio berhenti dengan kode {return_code}; periksa log.")
    except KeyboardInterrupt:
        print("Menghentikan Studio dan tunnel...")
    finally:
        stop_process(supervisor)
else:
    print("Studio dilewati. Ubah START_STUDIO=True untuk menjalankannya.")


## 9. Hentikan server dan simpan hasil

Setelah selesai, jalankan sel ini untuk membebaskan VRAM. File log dan hasil tetap disimpan. Untuk mencoba context lebih kecil setelah OOM, hentikan server, ubah `CONTEXT_LIMIT` di sel konfigurasi (misalnya 8192), jalankan ulang konfigurasi dan Start; jumlah token prompt + output harus tetap muat.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/kaggle_vllm.py", "stop"],
    cwd=PROJECT_DIR, check=True,
)
for path in (workspace / "vllm-eval-short.json", PROJECT_DIR / "logs/kaggle-vllm.log",
             PROJECT_DIR / "logs/kaggle-vllm-import.log"):
    if path.is_file():
        print(f"{path} ({path.stat().st_size:,} bytes)")
print("Simpan hasil dari Kaggle Files/Output. Tinjau log pihak ketiga sebelum membagikannya.")
